# Topic: Window Function Ranking

## Definition (30-second explanation)
* Window functions compute values across a set of rows related to the current row, without collapsing them into a single output row like `GROUP BY` does.
* Ranking window functions—`ROW_NUMBER()`, `RANK()`, and `DENSE_RANK()`—assign positional numbers to rows within a defined partition based on a specific order.

## Why Interviewers Ask This
* To test your ability to write sophisticated analytical queries (like Top-N reporting) without relying on inefficient self-joins.
* To verify you know how to deduplicate messy datasets safely.
* To see if you understand the precise logical execution order of SQL (e.g., filtering window functions requires a CTE).

## Core Concepts
* **OVER():** Defines the window. Read it as "for each group, in this order."
* **PARTITION BY:** Divides the result set into groups. Without it, the function computes across the entire table.
* **ORDER BY:** Defines the logical order of evaluation within each partition.
* **Execution Order:** Window functions are evaluated *after* `WHERE` and `GROUP BY` clauses.

## When to Use
* **ROW_NUMBER():** Best for deduplication, pagination, or when you need a strict, arbitrary tie-breaker.
* **RANK():** Best for leaderboards where gaps in ranking are mathematically required after ties.
* **DENSE_RANK():** Best for "Top-N" queries (e.g., Top 3 salaries) where all tied individuals should be included without skipping the next rank.
* **NTILE(n):** Best for percentile bucketing (e.g., dividing populations into quartiles/deciles).

## Advantages
* Keeps the row-level detail intact while providing aggregate/ranked insights.
* Significantly more readable and computationally efficient than correlated subqueries.

## Limitations
* Cannot be used directly inside a `WHERE` clause; requires wrapping in a Common Table Expression (CTE) or subquery to filter by the rank.

## Common Comparisons
| Function | Handles Ties By | Skips Next Number? | Best Use Case |
| :--- | :--- | :--- | :--- |
| **ROW_NUMBER()** | Giving unique incremental numbers | No | Deduplication, Pagination |
| **RANK()** | Giving the exact same rank | Yes (e.g., 1, 2, 2, 4) | Leaderboards |
| **DENSE_RANK()** | Giving the exact same rank | No (e.g., 1, 2, 2, 3) | Top-N reporting |

## Common Interview Traps
* **Forgetting PARTITION BY:** Causes the ranking to apply globally across the whole table instead of within the intended groups.
* **Using ROW_NUMBER for Top-N:** If two users tie for 3rd place, `ROW_NUMBER` arbitrarily cuts one off. Always use `DENSE_RANK` for Top-N unless instructed otherwise.
* **Filtering in the WHERE clause:** Writing `WHERE ROW_NUMBER() = 1` directly will throw a syntax error.

## Python / SQL Syntax
```sql
-- Standard Syntax Pattern
SELECT 
    department,
    salary,
    ROW_NUMBER() OVER(PARTITION BY department ORDER BY salary DESC) as rn,
    RANK() OVER(PARTITION BY department ORDER BY salary DESC) as rnk,
    DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) as dense_rnk
FROM employees;
```

## 45-Second Interview Answer
"Window function rankings assign sequential numbers to rows within specific partitions without collapsing the dataset. The three main functions handle ties differently: ROW_NUMBER always gives a unique integer, which is perfect for deduplication. RANK gives tied rows the same number but skips the subsequent numbers, useful for competitive leaderboards. DENSE_RANK gives tied rows the same number without skipping, making it the standard choice for Top-N analytical queries. Because window functions evaluate after the WHERE clause, I always wrap them in a CTE when I need to filter on the resulting rank."

## Practice Questions:

### Q1: Find the top 3 highest earners in each department.
* **Answer:** 
```sql
WITH ranked_employees AS (
    SELECT 
        name, department, salary,
        DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) as salary_rank
    FROM employees
)
SELECT name, department, salary, salary_rank
FROM ranked_employees
WHERE salary_rank <= 3;
```

**Common Mistakes:** Using ROW_NUMBER() instead of DENSE_RANK(), which would unfairly exclude tied employees from the Top 3. Trying to put WHERE DENSE_RANK() OVER(...) <= 3 directly in the main query instead of using a CTE.

**Likely Follow-up:** "How would you modify this query if we only wanted the exact top 3 rows per department, regardless of ties?" (Answer: Switch to ROW_NUMBER()).

### Q2:
**Scenario:**
You are analyzing customer purchasing behavior using the `classicmodels` database. You need to extract a clean list showing only the *single most recent order* placed by each customer. 

Sometimes, a customer places multiple orders on the exact same date. In those cases, you must break the tie by selecting the order with the highest `orderNumber`.

**Data Requirement (Mock Schema from `classicmodels`):**
Table: `orders`
Columns: 
* `orderNumber` (INT, Primary Key)
* `orderDate` (DATE)
* `customerNumber` (INT)

**Question:** 
Write a SQL query to return the `customerNumber`, `orderDate`, and `orderNumber` for the single most recent order per customer. You must ensure exactly one row is returned per customer, even if there are ties on the date.

* **Ideal Interview Answer:** 
```sql
WITH RankedOrders AS (
    SELECT 
        customerNumber, 
        orderDate, 
        orderNumber,
        ROW_NUMBER() OVER(
            PARTITION BY customerNumber 
            ORDER BY orderDate DESC, orderNumber DESC -- Secondary sort critical for deterministic tie-breaking
        ) as rn
    FROM orders
)
SELECT 
    customerNumber, 
    orderDate, 
    orderNumber
FROM RankedOrders
WHERE rn = 1;
```
* **Common Mistakes:** 
    * Forgetting the secondary sort column in the `ORDER BY`, which leads to arbitrary/random row selection when ties occur. 
    * Using `RANK()` instead of `ROW_NUMBER()`, which would return duplicate rows if a tie happened.
* **Interview Tip:** Whenever an interviewer asks you to "deduplicate" data or find the "latest status," immediately reach for `ROW_NUMBER() = 1` inside a CTE. Always ensure your `ORDER BY` is strict enough to prevent random tie-breaking.

### Q3:
**You are a Data Scientist analyzing customer lifetime value. You want to divide your entire customer base into exactly 4 equal groups (quartiles) based on their total_spent, where Quartile 1 contains the highest spenders.**

Data Requirement:
Table: customer_spend
Columns: customer_id, total_spent

**Write a SQL query that returns the customer_id, total_spent, and their assigned quartile number.**

**Mock Schema:**
```sql
-- DDL
CREATE TABLE customer_spend (
    customer_id INT PRIMARY KEY,
    total_spent DECIMAL(10, 2)
);

-- Mock Data Insertion
INSERT INTO customer_spend (customer_id, total_spent) VALUES
(101, 5000.00), 
(102, 4500.50), 
(103, 3000.00), 
(104, 2500.00), 
(105, 1200.75), 
(106, 1000.00), 
(107, 500.00),  
(108, 150.00);
```

**Answer:** 
```sql
SELECT 
    customer_id, 
    total_spent,
    NTILE(4) OVER(ORDER BY total_spent DESC) AS quartile
FROM customer_spend;
```
* **Common Mistakes:** 
    * Using `ORDER BY total_spent ASC` (the default if `DESC` is omitted), which would put the *lowest* spenders into Quartile 1.
    * Trying to manually calculate total rows and divide by 4 using complex subqueries instead of utilizing the built-in `NTILE()` function.
* **Interview Tip:** If an interviewer asks what happens when the number of rows doesn't divide perfectly by 4 (e.g., 9 rows into 4 buckets), you can impress them by knowing that `NTILE` puts the extra rows into the first buckets (so the sizes would be 3, 2, 2, 2).